# Hyperparameter Tuning & Advanced Models

## Objectives

- Train advanced machine learning models
- Use Stratified K-Fold Cross Validation
- Track experiments with MLflow
- Tune XGBoost and CatBoost using Optuna
- Compare model performance
- Select the best model

Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import joblib

Load Dataset

In [2]:
df = pd.read_csv("../data/processed/clustered_data.csv")

df["y"] = df["y"].map({
    "no":0,
    "yes":1
})

X = df.drop("y", axis=1)

y = df["y"]

Train/Test Split

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    stratify=y,

    random_state=42
)

Load Preprocessor

In [4]:
preprocessor = joblib.load(
    "../models/preprocessor.pkl"
)

print("Preprocessor Loaded Successfully.")

Preprocessor Loaded Successfully.


Stratified K-Fold

In [5]:
cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

Start MLflow

In [6]:
mlflow.set_experiment("Marketing Campaign Response Modelling")

<Experiment: artifact_location=('file:///d:/Important Files/Marketing Campaign Response '
 'Modelling/notebooks/mlruns/1'), creation_time=1782966817289, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782966817289, lifecycle_stage='active', name='Marketing Campaign Response Modelling', tags={}, trace_location=None, workspace='default'>

XGBoost Pipeline

In [7]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),

    ("classifier",
     XGBClassifier(

         random_state=42,

         eval_metric="logloss",

         use_label_encoder=False
     ))
])

xgb_pipeline.fit(X_train, y_train)

print("XGBoost Model Trained Successfully.")

XGBoost Model Trained Successfully.


XGBoost Predictions

In [8]:
y_pred_xgb = xgb_pipeline.predict(X_test)

y_prob_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]

XGBoost Evaluation

In [9]:
xgb_results = {

    "Model": "XGBoost",

    "Accuracy (%)": accuracy_score(y_test, y_pred_xgb) * 100,

    "Precision (%)": precision_score(y_test, y_pred_xgb) * 100,

    "Recall (%)": recall_score(y_test, y_pred_xgb) * 100,

    "F1 Score (%)": f1_score(y_test, y_pred_xgb) * 100,

    "ROC AUC (%)": roc_auc_score(y_test, y_prob_xgb) * 100
}

pd.DataFrame([xgb_results]).round(2)

,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%),ROC AUC (%)
0,XGBoost,90.03,61.71,30.39,40.72,80.17


XGBoost Cross Validation

In [10]:
xgb_cv = cross_val_score(

    xgb_pipeline,

    X,

    y,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1
)

print(f"Mean ROC AUC (%): {xgb_cv.mean()*100:.2f}")

print(f"Std (%): {xgb_cv.std()*100:.2f}")

Mean ROC AUC (%): 78.94
Std (%): 0.87


MLflow Logging (XGBoost)

In [11]:
with mlflow.start_run(run_name="XGBoost_Baseline"):

    mlflow.log_param("Model", "XGBoost")

    mlflow.log_metric("Accuracy", xgb_results["Accuracy (%)"])
    mlflow.log_metric("Precision", xgb_results["Precision (%)"])
    mlflow.log_metric("Recall", xgb_results["Recall (%)"])
    mlflow.log_metric("F1_Score", xgb_results["F1 Score (%)"])
    mlflow.log_metric("ROC_AUC", xgb_results["ROC AUC (%)"])

print("XGBoost logged successfully.")

XGBoost logged successfully.


Save XGBoost Model

In [12]:
joblib.dump(
    xgb_pipeline,
    "../models/xgboost_model.pkl"
)

print("XGBoost model saved.")

XGBoost model saved.


Evaluation Function

In [13]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    results = {

        "Accuracy (%)": accuracy_score(y_test, y_pred) * 100,

        "Precision (%)": precision_score(y_test, y_pred) * 100,

        "Recall (%)": recall_score(y_test, y_pred) * 100,

        "F1 Score (%)": f1_score(y_test, y_pred) * 100,

        "ROC AUC (%)": roc_auc_score(y_test, y_prob) * 100

    }

    return results

LightGBM Pipeline

In [14]:
lgbm_pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("classifier",

     LGBMClassifier(

         random_state=42,

         verbose=-1

     ))

])

lgbm_pipeline.fit(X_train, y_train)

print("LightGBM trained successfully.")

LightGBM trained successfully.


Evaluate LightGBM

In [15]:
lgbm_results = evaluate_model(

    lgbm_pipeline,

    X_test,

    y_test

)

lgbm_results["Model"] = "LightGBM"

pd.DataFrame([lgbm_results]).round(2)

,Accuracy (%),Precision (%),Recall (%),F1 Score (%),ROC AUC (%),Model
0,90.18,65.45,27.16,38.39,81.09,LightGBM


Cross Validation

In [16]:
lgbm_cv = cross_val_score(

    lgbm_pipeline,

    X,

    y,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1

)

print(f"Mean ROC AUC (%): {lgbm_cv.mean()*100:.2f}")

print(f"Std (%): {lgbm_cv.std()*100:.2f}")

Mean ROC AUC (%): 80.11
Std (%): 0.71


MLflow Logging

In [17]:
with mlflow.start_run(run_name="LightGBM_Baseline"):

    mlflow.log_param("Model", "LightGBM")

    mlflow.log_metric("Accuracy", lgbm_results["Accuracy (%)"])
    mlflow.log_metric("Precision", lgbm_results["Precision (%)"])
    mlflow.log_metric("Recall", lgbm_results["Recall (%)"])
    mlflow.log_metric("F1_Score", lgbm_results["F1 Score (%)"])
    mlflow.log_metric("ROC_AUC", lgbm_results["ROC AUC (%)"])

print("LightGBM logged successfully.")

LightGBM logged successfully.


Save Model

In [18]:
joblib.dump(

    lgbm_pipeline,

    "../models/lightgbm_model.pkl"

)

print("LightGBM Saved.")

LightGBM Saved.


CatBoost Pipeline

In [19]:
cat_pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("classifier",

     CatBoostClassifier(

         random_state=42,

         verbose=False

     ))

])

cat_pipeline.fit(

    X_train,

    y_train

)

print("CatBoost trained successfully.")

CatBoost trained successfully.


Evaluate CatBoost

In [20]:
cat_results = evaluate_model(

    cat_pipeline,

    X_test,

    y_test

)

cat_results["Model"] = "CatBoost"

pd.DataFrame([cat_results]).round(2)

,Accuracy (%),Precision (%),Recall (%),F1 Score (%),ROC AUC (%),Model
0,90.09,63.27,28.77,39.56,81.23,CatBoost


Cross Validation

In [21]:
cat_cv = cross_val_score(

    cat_pipeline,

    X,

    y,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1

)

print(f"Mean ROC AUC (%): {cat_cv.mean()*100:.2f}")

print(f"Std (%): {cat_cv.std()*100:.2f}")

Mean ROC AUC (%): 79.96
Std (%): 0.63


CatBoost MLflow

In [22]:
with mlflow.start_run(run_name="CatBoost_Baseline"):

    mlflow.log_param("Model", "CatBoost")

    mlflow.log_metric("Accuracy", cat_results["Accuracy (%)"])
    mlflow.log_metric("Precision", cat_results["Precision (%)"])
    mlflow.log_metric("Recall", cat_results["Recall (%)"])
    mlflow.log_metric("F1_Score", cat_results["F1 Score (%)"])
    mlflow.log_metric("ROC_AUC", cat_results["ROC AUC (%)"])

print("CatBoost logged successfully.")

CatBoost logged successfully.


Save Model

In [23]:
joblib.dump(

    cat_pipeline,

    "../models/catboost_model.pkl"

)

print("CatBoost Saved.")

CatBoost Saved.


Stacking Ensemble

In [24]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stacking_model = StackingClassifier(

    estimators=[
        ("xgb", xgb_pipeline),
        ("lgbm", lgbm_pipeline),
        ("cat", cat_pipeline)
    ],

    final_estimator=LogisticRegression(max_iter=1000),

    cv=5,
    n_jobs=-1
)

stacking_model.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('xgb', ...), ('lgbm', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...max_iter=1000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,) or list of ndarray if `y` is of type `""multilabel-indicator""`.Class labels.","ndarray[int64](2,)","[0,1]"
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[Pipeline(step...=None, ...))]), Pipeline(step...verbose=-1))]), Pipeline(step...bose=False))])]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimators expose such an attribute when fit... versionadde

In [25]:
# Save train-test split for next notebook

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_frame(name="y").to_csv("../data/processed/y_train.csv", index=False)
y_test.to_frame(name="y").to_csv("../data/processed/y_test.csv", index=False)

print("Train/Test datasets saved successfully.")

Train/Test datasets saved successfully.


In [26]:
import joblib

# Save CatBoost model as final model
joblib.dump(cat_pipeline, "../models/best_model.pkl")

print("Best model saved successfully!")

Best model saved successfully!
